<img src="https://hilpisch.com/tpq_logo_bic.png" width="20%" align="right">

# DXQuant · Derivatives Analytics with Python
## Notebook 11 · Interpolation and Curve Queries

&copy; Dr. Yves J. Hilpisch<br>
AI-Powered by different LLMs<br>
The Python Quants GmbH | https://tpq.io<br>
https://hilpisch.com | https://linktr.ee/dyjh

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yhilpisch/dxquant/blob/main/notebooks/11_dxquant_interpolation_and_curves.ipynb)


## Notebook Goals

This notebook focuses on deterministic building blocks used by many valuation routines:

- one-dimensional interpolation (`linear_interpolate`, `Spline1D`), and
- curve queries (`InterpolatedZeroCurve`): zero rates, discount factors, forwards.

The examples use continuously compounded zero rates for simplicity.

## Setup

In Google Colab, select **Runtime → Run all** with a standard CPU runtime. This cell installs DXQuant and Matplotlib if they are missing. No GPU, Google Drive mount, or API credentials are needed. The examples use synthetic inputs.

Before the first PyPI release, installation uses the public GitHub `main` branch. When running from a local checkout, the setup cell installs the local package instead.


In [ ]:
from __future__ import annotations

import importlib.util
import subprocess
import sys
from pathlib import Path

requirements = []
if importlib.util.find_spec("dxquant") is None:
    package_source = (
        "dxquant @ git+https://github.com/yhilpisch/dxquant.git@main"
    )
    for directory in (Path.cwd(), Path.cwd().parent):
        local_package = directory
        if (local_package / "pyproject.toml").is_file():
            package_source = str(local_package.resolve())
            break
    requirements.append(package_source)
if importlib.util.find_spec("matplotlib") is None:
    requirements.append("matplotlib>=3.8")
if requirements:
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "--quiet", *requirements]
    )


In [ ]:
import datetime as dt

import numpy as np

import dxquant as dx

np.set_printoptions(precision=6, suppress=True)

ENABLE_PLOTS = False
plt = None
if ENABLE_PLOTS:
    try:
        import matplotlib.pyplot as plt
    except ModuleNotFoundError:
        plt = None

## 1. Linear Interpolation

Linear interpolation is often adequate for quick prototypes and for debugging curve
construction.


In [ ]:
x_nodes = np.array([0.0, 1.0, 2.0, 3.0])
y_nodes = np.array([0.02, 0.025, 0.027, 0.028])

x_query = np.array([0.5, 1.5, 2.5])
y_query = dx.linear_interpolate(x_nodes, y_nodes, x_query)
y_query

## 2. Spline Interpolation

Spline interpolation provides smooth values and derivatives. DXQuant exposes a small wrapper with
explicit validation.


In [ ]:
spline = dx.Spline1D(x_nodes, y_nodes, degree=3)
values = spline.value(x_query)
derivs = spline.derivative(x_query)
values, derivs

## 3. Interpolated Zero Curve

An `InterpolatedZeroCurve` stores (date, zero rate) nodes and uses spline interpolation for
queries between nodes.


In [ ]:
pricing_date = dt.date(2024, 1, 1)

nodes = [
    (dt.date(2024, 7, 1), 0.0200),
    (dt.date(2025, 1, 1), 0.0225),
    (dt.date(2026, 1, 1), 0.0250),
    (dt.date(2027, 1, 1), 0.0265),
]

curve = dx.InterpolatedZeroCurve(
    name="zc",
    reference_date=pricing_date,
    nodes=nodes,
)

query = dt.date(2025, 7, 1)
print("zero:", curve.zero_rate(query))
print("df:", curve.discount_factor(query))

## 4. Forward Rates

Forward rates are derived from discount factors. Smooth interpolation also allows an
instantaneous forward via the spline derivative.


In [ ]:
start = dt.date(2025, 1, 1)
end = dt.date(2026, 1, 1)

f = curve.forward_rate(start, end)
inst = curve.instantaneous_forward(query)

print("forward rate:", f)
print("inst forward:", inst)

## 5. Optional Visualization

If you enable plots, you can inspect the node rates and the interpolated curve.

In [ ]:
if plt is not None:
    dates = [d for d, _ in nodes]
    zr_nodes = [r for _, r in nodes]
    base = dt.date(2024, 1, 1)
    grid_dates = [base + dt.timedelta(days=30 * k) for k in range(1, 37)]
    zr_grid = [curve.zero_rate(d) for d in grid_dates]

    plt.figure(figsize=(10, 4))
    plt.plot(dates, zr_nodes, marker="o", label="nodes")
    plt.plot(grid_dates, zr_grid, label="interp")
    plt.title("Interpolated zero curve")
    plt.xlabel("date")
    plt.ylabel("zero rate")
    plt.legend()
    plt.show()

<img src="https://hilpisch.com/tpq_logo_bic.png" width="20%" align="right">